# 売上の悪い商品を特定するためのEDA

`sample_data/orders.csv`（注文データ、800件）を対象に、商品ごとの売れ行きを様々な角度から確認し、売上が伸びていない商品・実質的な売上に結びついていない商品を特定するための探索的データ分析（EDA）を行う。

列構成は`README.md`のデータセット定義（order_id, user_id, product_id, product_name, category, quantity, unit_price, discount_rate, sale_price, status, payment_method, created_at, shipped_at）に基づく。

## キャンセル・返品の扱い（分析方針）

- `status = "Cancelled"`（取引未成立）の注文は、以降の売上・数量集計から**完全に除外**する。
- `status = "Returned"`（一度購入後に返品）の注文は、決済自体は発生しているが最終的に売上として残らない。そのため以下の2種類の売上を区別して扱う。
  - **グロス売上**: `Cancelled`のみ除外（`Returned`は含む）。実際に決済が発生した金額。
  - **ネット売上**: `Cancelled`と`Returned`の両方を除外。最終的に売上として残った金額。
  - あわせて商品別の**返品率**（返品件数 / 成立注文件数）を算出し、返品率が高い商品も「売上が悪い（実質的に売上に繋がっていない）」商品の一種として扱う。
- `Complete` / `Shipped` / `Processing` は正常に成立した注文として扱う。

## 1. データの読み込みと構造確認

このセルの目的: `orders.csv`を読み込み、`README.md`に記載の列構成・件数（800件）と実データが一致しているかを確認する。

In [1]:
import pandas as pd
import plotly.express as px

orders = pd.read_csv(
    "../sample_data/orders.csv",
    parse_dates=["created_at", "shipped_at"],
)
print(orders.shape)
orders.head()

(800, 13)


,order_id,user_id,product_id,product_name,category,quantity,unit_price,discount_rate,sale_price,status,payment_method,created_at,shipped_at
0,1,133,108,Wool Sweater,Apparel,1,69.0,0.2,55.2,Complete,mobile_payment,2025-01-01 09:42:00,2025-01-03 09:42:00
1,2,58,107,Running Shoes,Apparel,1,129.0,0.0,129.0,Complete,bank_transfer,2025-01-02 16:43:00,2025-01-03 16:43:00
2,3,31,104,USB-C Hub,Electronics,1,39.0,0.0,39.0,Complete,mobile_payment,2025-01-02 23:54:00,2025-01-07 23:54:00
3,4,79,111,Air Fryer,Home & Kitchen,1,149.0,0.0,149.0,Complete,mobile_payment,2025-01-10 03:13:00,2025-01-12 03:13:00
4,5,36,122,Board Game,Toys,1,39.0,0.0,39.0,Cancelled,credit_card,2025-01-10 13:53:00,NaT


このセルの目的: 各列の型と欠損値の有無を確認する（`shipped_at`は未出荷の場合に欠損することがREADMEに記載されているため、欠損があること自体は想定通りか確認する）。

In [2]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        800 non-null    int64         
 1   user_id         800 non-null    int64         
 2   product_id      800 non-null    int64         
 3   product_name    800 non-null    str           
 4   category        800 non-null    str           
 5   quantity        800 non-null    int64         
 6   unit_price      800 non-null    float64       
 7   discount_rate   800 non-null    float64       
 8   sale_price      800 non-null    float64       
 9   status          800 non-null    str           
 10  payment_method  800 non-null    str           
 11  created_at      800 non-null    datetime64[us]
 12  shipped_at      681 non-null    datetime64[us]
dtypes: datetime64[us](2), float64(3), int64(4), str(4)
memory usage: 112.2 KB


## 2. ステータスの内訳確認

このセルの目的: `status`列の値の種類と件数を確認し、キャンセル・返品がそれぞれどの程度含まれているかを把握する。

In [3]:
orders["status"].value_counts()

status
Complete      593
Cancelled      84
Shipped        57
Processing     35
Returned       31
Name: count, dtype: int64

## 3. 分析対象データの定義（キャンセル・返品の扱いを反映）

このセルの目的: 上記の方針に基づき、`Cancelled`のみ除外した`orders_gross`（グロス売上用）と、さらに`Returned`も除外した`orders_net`（ネット売上用）の2つのデータフレームを作成する。以降の集計は基本的に`orders_net`を用いる。

In [4]:
orders_gross = orders[orders["status"] != "Cancelled"].copy()
orders_net = orders_gross[orders_gross["status"] != "Returned"].copy()

print("全件数:", len(orders))
print("gross（Cancelled除外）:", len(orders_gross))
print("net（Cancelled・Returned除外）:", len(orders_net))

全件数: 800
gross（Cancelled除外）: 716
net（Cancelled・Returned除外）: 685


## 4. 商品別の販売数量・売上金額の分布（ヒストグラム）

このセルの目的: `orders_net`を用いて商品ごとの合計販売数量・合計売上・注文件数を集計し、以降の分析で使う`product_summary`を作成する。

In [5]:
product_summary = (
    orders_net.groupby(["product_id", "product_name", "category"], as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("sale_price", "sum"),
        order_count=("order_id", "count"),
    )
)
product_summary.sort_values("total_sales", ascending=False).head()

,product_id,product_name,category,total_quantity,total_sales,order_count
17,118,Camping Tent,Sports,56,13769.7,37
1,102,Smart Watch,Electronics,51,9651.5,35
10,111,Air Fryer,Home & Kitchen,50,7286.1,29
16,117,Dumbbell Set,Sports,42,6344.1,27
6,107,Running Shoes,Apparel,43,5327.7,29


このセルの目的: 商品別の合計販売数量の分布をヒストグラムで確認し、少数の商品に販売数量が偏っているか（ロングテールになっているか）を見る。

In [6]:
fig_qty_hist = px.histogram(
    product_summary, x="total_quantity", nbins=20, title="商品別 合計販売数量の分布（ネット）"
)
fig_qty_hist.update_layout(xaxis_title="商品別合計販売数量", yaxis_title="商品数")
fig_qty_hist.show()

このセルの目的: 商品別の合計売上金額の分布をヒストグラムで確認し、売上が低い商品がどの価格帯に集中しているかを見る。

In [7]:
fig_sales_hist = px.histogram(
    product_summary, x="total_sales", nbins=20, title="商品別 合計売上金額の分布（ネット）"
)
fig_sales_hist.update_layout(xaxis_title="商品別合計売上 (USD)", yaxis_title="商品数")
fig_sales_hist.show()

## 5. 商品別売上ランキング（上位・下位）

このセルの目的: ネット売上の合計金額で商品をランキングし、売上上位10商品を確認する。

In [8]:
top10 = product_summary.sort_values("total_sales", ascending=False).head(10)
fig_top10 = px.bar(
    top10, x="total_sales", y="product_name", orientation="h", title="売上上位10商品（ネット）"
)
fig_top10.update_layout(
    xaxis_title="売上 (USD)", yaxis_title="商品名", yaxis={"categoryorder": "total ascending"}
)
fig_top10.show()

このセルの目的: 売上下位10商品を確認し、売上が悪い商品の候補を特定する。

In [9]:
bottom10 = product_summary.sort_values("total_sales", ascending=True).head(10)
fig_bottom10 = px.bar(
    bottom10, x="total_sales", y="product_name", orientation="h", title="売上下位10商品（ネット）"
)
fig_bottom10.update_layout(
    xaxis_title="売上 (USD)", yaxis_title="商品名", yaxis={"categoryorder": "total descending"}
)
fig_bottom10.show()

## 6. カテゴリ別の売上構成

このセルの目的: カテゴリごとの売上合計とシェアを確認し、どのカテゴリが売上を牽引し、どのカテゴリが弱いかを把握する。

In [10]:
category_summary = (
    product_summary.groupby("category", as_index=False)["total_sales"]
    .sum()
    .sort_values("total_sales", ascending=False)
)
fig_category = px.pie(
    category_summary, names="category", values="total_sales", title="カテゴリ別売上構成（ネット）"
)
fig_category.show()

## 7. 返品率が高い商品の特定（実質的に売上に結びついていない商品）

このセルの目的: 商品別に返品率（返品件数 / `Cancelled`除外後の全注文件数）を計算する。返品率が高い商品は、一時的な売上（グロス）が発生していても最終的なネット売上には結びついていないため、「売上が悪い商品」として扱う対象になる。

In [11]:
returned_by_product = (
    orders_gross.assign(is_returned=orders_gross["status"] == "Returned")
    .groupby(["product_id", "product_name"], as_index=False)
    .agg(order_count=("order_id", "count"), returned_count=("is_returned", "sum"))
)
returned_by_product["return_rate"] = (
    returned_by_product["returned_count"] / returned_by_product["order_count"]
)
returned_by_product.sort_values("return_rate", ascending=False).head(10)

,product_id,product_name,order_count,returned_count,return_rate
15,116,Yoga Mat,41,5,0.121951
7,108,Wool Sweater,41,5,0.121951
16,117,Dumbbell Set,29,2,0.068966
8,109,Ceramic Mug Set,30,2,0.066667
13,114,Sunscreen SPF50,33,2,0.060606
1,102,Smart Watch,37,2,0.054054
18,119,Mystery Novel,19,1,0.052632
21,122,Board Game,38,2,0.052632
17,118,Camping Tent,39,2,0.051282
5,106,Denim Jacket,24,1,0.041667


## 8. まとめ: 売上が悪い商品の候補リスト

このセルの目的: 「ネット売上が低い商品」と「返品率が高い商品」を1つの表に統合し、売上改善のアクションを検討すべき商品の最終的な候補リストを作成する。

In [12]:
low_sales_candidates = product_summary.merge(
    returned_by_product[["product_id", "return_rate"]], on="product_id", how="left"
)
low_sales_candidates = low_sales_candidates.sort_values(
    ["total_sales", "return_rate"], ascending=[True, False]
)
low_sales_candidates.head(10)

,product_id,product_name,category,total_quantity,total_sales,order_count,return_rate
18,119,Mystery Novel,Books,33,443.8,18,0.052632
11,112,Scented Candle,Home & Kitchen,52,763.5,35,0.027778
4,105,Cotton T-Shirt,Apparel,45,849.3,30,0.000000
13,114,Sunscreen SPF50,Beauty,54,1155.0,31,0.060606
19,120,Cookbook,Books,46,1430.4,30,0.032258
8,109,Ceramic Mug Set,Home & Kitchen,51,1432.6,28,0.066667
12,113,Vitamin C Serum,Beauty,46,1533.0,31,0.031250
3,104,USB-C Hub,Electronics,47,1778.4,35,0.027778
21,122,Board Game,Toys,70,2663.7,36,0.052632
15,116,Yoga Mat,Sports,63,2722.5,36,0.121951


## 9. 価格帯（unit_priceの四分位）別の販売数量比較

このセルの目的: 商品の`unit_price`（商品単価）を四分位で4つの価格帯に区切り、価格帯ごとの合計販売数量（ネット）を集計する。単価が高い商品ほど数量が出にくいといった傾向があるかを確認する。

In [13]:
product_price = orders.drop_duplicates("product_id")[["product_id", "unit_price"]]
product_price["price_band"] = pd.qcut(
    product_price["unit_price"], q=4, labels=["Q1(低価格)", "Q2", "Q3", "Q4(高価格)"]
)

price_band_summary = product_summary.merge(
    product_price[["product_id", "price_band"]], on="product_id"
)
price_band_quantity = price_band_summary.groupby("price_band", as_index=False)[
    "total_quantity"
].sum()
price_band_quantity

,price_band,total_quantity
0,Q1(低価格),281
1,Q2,298
2,Q3,236
3,Q4(高価格),272


このセルの目的: 価格帯別の合計販売数量を棒グラフで可視化し、価格帯と販売数量の関係を見やすくする。

In [14]:
fig_price_band = px.bar(
    price_band_quantity,
    x="price_band",
    y="total_quantity",
    title="価格帯別 合計販売数量（ネット）",
)
fig_price_band.update_layout(
    xaxis_title="価格帯（unit_priceの四分位）", yaxis_title="合計販売数量"
)
fig_price_band.show()

## 10. カテゴリ内での販売数量の相対順位

このセルの目的: 同一カテゴリ内で商品の合計販売数量（ネット）の順位・パーセンタイルを計算する。カテゴリごとに商品数や市場規模が異なるため、全体順位ではなくカテゴリ内での相対的な位置づけを見る。

In [15]:
category_rank = product_summary.copy()
category_rank["category_size"] = category_rank.groupby("category")["product_id"].transform(
    "count"
)
category_rank["rank_in_category"] = category_rank.groupby("category")["total_quantity"].rank(
    method="min", ascending=True
)
category_rank["percentile_in_category"] = category_rank.groupby("category")[
    "total_quantity"
].rank(pct=True, ascending=True)

category_rank.sort_values(["category", "rank_in_category"])[
    [
        "category",
        "product_name",
        "total_quantity",
        "rank_in_category",
        "category_size",
        "percentile_in_category",
    ]
]

,category,product_name,total_quantity,rank_in_category,category_size,percentile_in_category
5,Apparel,Denim Jacket,41,1.0,4,0.250000
6,Apparel,Running Shoes,43,2.0,4,0.500000
4,Apparel,Cotton T-Shirt,45,3.0,4,0.750000
7,Apparel,Wool Sweater,56,4.0,4,1.000000
14,Beauty,Hair Dryer,30,1.0,3,0.333333
12,Beauty,Vitamin C Serum,46,2.0,3,0.666667
13,Beauty,Sunscreen SPF50,54,3.0,3,1.000000
18,Books,Mystery Novel,33,1.0,2,0.500000
19,Books,Cookbook,46,2.0,2,1.000000
0,Electronics,Wireless Earbuds,40,1.0,4,0.250000


このセルの目的: カテゴリ内下位20%（カテゴリごとの販売数量の20パーセンタイル値以下）に該当する商品を一覧化する。今回のデータはカテゴリあたりの商品数が2〜4個と少ないため、下位20%は実質的に「カテゴリ内で最も販売数量が少ない商品（1〜2商品）」を指すことになる点に注意する。

In [16]:
category_rank["category_p20_threshold"] = category_rank.groupby("category")[
    "total_quantity"
].transform(lambda s: s.quantile(0.2))

bottom20_by_category = category_rank[
    category_rank["total_quantity"] <= category_rank["category_p20_threshold"]
].sort_values(["category", "total_quantity"])

bottom20_by_category[
    ["category", "product_name", "total_quantity", "percentile_in_category", "category_size"]
]

,category,product_name,total_quantity,percentile_in_category,category_size
5,Apparel,Denim Jacket,41,0.250000,4
14,Beauty,Hair Dryer,30,0.333333,3
18,Books,Mystery Novel,33,0.500000,2
0,Electronics,Wireless Earbuds,40,0.250000,4
9,Home & Kitchen,Chef Knife,39,0.250000,4
16,Sports,Dumbbell Set,42,0.333333,3
21,Toys,Board Game,70,0.500000,2


## まとめ

- ネット売上（`Cancelled`・`Returned`を除外した売上）が低い商品と、返品率が高い商品を突き合わせることで、「そもそも売れていない商品」と「売れているが返品されて実質的な売上に繋がっていない商品」を区別しながら特定できる。
- 上記の`low_sales_candidates`の上位（売上が低く、かつ返品率が高い）商品が、売上改善のアクションを優先的に検討すべき商品の候補となる。
- 価格帯（`unit_price`の四分位）別に見ると、価格帯によって合計販売数量に差があり、高価格帯の商品は数量が出にくい傾向を確認できる（`price_band_quantity`）。
- カテゴリ内での相対順位（`category_rank`）を見ると、カテゴリごとに市場規模・商品数が異なるため、全体順位だけでは見えない「カテゴリ内で相対的に弱い商品」（`bottom20_by_category`）を特定できる。ただし本データはカテゴリあたり商品数が少ないため、下位20%は実質的にカテゴリ最下位1商品程度になる点に留意する。